
# Customer Churn Prediction using Artificial Neural Network (ANN)

## Project Overview
This project predicts whether a customer is likely to churn using a Deep Learning model built with TensorFlow/Keras.

### Features Used
- Credit Score
- Geography
- Gender
- Age
- Tenure
- Balance
- Number of Products
- Credit Card Ownership
- Active Membership Status
- Estimated Salary

### Technologies Used
- Python
- Pandas & NumPy
- TensorFlow / Keras
- Scikit-learn
- Streamlit

### Workflow
1. Data Preprocessing
2. Feature Encoding
3. Feature Scaling
4. ANN Model Training
5. Model Evaluation
6. Streamlit Deployment


In [22]:
# Feature Scaling
# Train-Test Split

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [23]:
# Load Dataset
data_csv = pd.read_csv("Churn_Modelling.csv")
# data_csv.head()

In [24]:
preprocess_data = data_csv.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [25]:
label_encoder_gender = LabelEncoder()
preprocess_data['Gender'] = label_encoder_gender.fit_transform(preprocess_data['Gender'])

In [26]:
# Encoding Categorical Variables
from sklearn.preprocessing import OneHotEncoder
onehot_encoding_geo = OneHotEncoder()
preprocess_geo_field = onehot_encoding_geo.fit_transform(preprocess_data[['Geography']])
preprocess_geo_field.toarray()
onehot_encode_df = pd.DataFrame(preprocess_geo_field.toarray(),columns=onehot_encoding_geo.get_feature_names_out(['Geography']))

In [27]:
preprocess_data = preprocess_data.drop('Geography',axis=1)

In [28]:
preprocess_data = pd.concat([preprocess_data,onehot_encode_df],axis=1)
preprocess_data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [29]:
with open('label_encoder_gender.pkl','wb') as f:
    pickle.dump(label_encoder_gender,f)

with open('onehot_encoding_geo.pkl','wb') as f:
    pickle.dump(onehot_encoding_geo,f)

In [30]:
# Feature Scaling
# Train-Test Split
x = preprocess_data.drop(['Exited'],axis=1)
y = preprocess_data['Exited']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [31]:
with open('scaler.pkl','wb') as f:
    pickle.dump(scaler,f)

ANN Implementation

In [32]:
# Build ANN Model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

In [33]:
# Build ANN Model
model = Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),
    Dense(34,activation='relu'),
    Dense(1,activation='sigmoid')]
)
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 34)                2210      
                                                                 
 dense_5 (Dense)             (None, 1)                 35        
                                                                 
Total params: 3077 (12.02 KB)
Trainable params: 3077 (12.02 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [34]:
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=['accuracy'])

In [35]:
# Train the Model
import datetime
log_dir = 'logs/fit' + datetime.datetime.now().strftime("%y%m%d")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)
early_stopping = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
history = model.fit(
    x_train,y_train,validation_data=(x_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping]
)


Epoch 1/100
250/250 [==============================] - 5s 11ms/step - loss: 0.4508 - accuracy: 0.8035 - val_loss: 0.3912 - val_accuracy: 0.8325
Epoch 2/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3861 - accuracy: 0.8376 - val_loss: 0.3559 - val_accuracy: 0.8465
Epoch 3/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3583 - accuracy: 0.8504 - val_loss: 0.3449 - val_accuracy: 0.8575
Epoch 4/100
250/250 [==============================] - 2s 6ms/step - loss: 0.3478 - accuracy: 0.8577 - val_loss: 0.3447 - val_accuracy: 0.8555
Epoch 5/100
250/250 [==============================] - 2s 6ms/step - loss: 0.3432 - accuracy: 0.8579 - val_loss: 0.3409 - val_accuracy: 0.8615
Epoch 6/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3376 - accuracy: 0.8595 - val_loss: 0.3394 - val_accuracy: 0.8620
Epoch 7/100
250/250 [==============================] - 2s 6ms/step - loss: 0.3354 - accuracy: 0.8639 - val_loss: 0.3407 - val_accuracy: 0.865

In [36]:
model.save("model.h5")

c:\Users\DELL\pdf-chatbot\.venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [37]:
%load_ext tensorboard

In [38]:
# %tensorboard --logdir logs/fit